# 03. Trip matching and snapshot rules

Joins GTFS-RT predictions to the static schedule to produce stop-event-level delay records, then deduplicates each scheduled stop event under three different rules.

The dedup choice is the key methodological lever this project examines. Same data, three defensible ways to pick which prediction represents the stop event:

- **`closest`**: prediction whose time is closest to the scheduled time. Standard in the v1 pipeline. Biases OTP upward because the rule selects on the same property OTP measures.
- **`last_before`**: last prediction issued before scheduled time. What a rider would have seen on the app right before the bus was due.
- **`latest`**: chronologically last prediction available. Closest GTFS-RT-only proxy for actual arrival, since predictions converge to actual time as the bus approaches.

**Prerequisites:** `gtfsrt`, `trips`, `calendar`, `calendar_dates`, `stop_times` from notebook 02.

**Outputs:** `matched_stops`, `trip_delays_closest`, `trip_delays_last_before`, `trip_delays_latest`.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r'C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework')
DB_PATH      = PROJECT_ROOT / 'data' / 'transit_kpi.duckdb'

con = duckdb.connect(str(DB_PATH))

print(f'Database: {DB_PATH}')
print()
print('Tables:')
print(con.sql('SHOW TABLES').df().to_string(index=False))

In [ ]:
required_tables = ['gtfsrt', 'trips', 'calendar', 'calendar_dates', 'stop_times']

existing = set(con.sql('SHOW TABLES').df()['name'].tolist())
missing = [t for t in required_tables if t not in existing]
if missing:
    raise RuntimeError(
        f'Missing required tables: {missing}. Run notebook 02 first.'
    )

print('All required tables present. Row counts:')
for tbl in required_tables:
    n = con.sql(f'SELECT COUNT(*) AS n FROM {tbl}').fetchone()[0]
    print(f'  {tbl:18s} {n:>14,} rows')

print()
print('gtfsrt service date range:')
print(con.sql("""
    SELECT
        MIN(service_date) AS first_date,
        MAX(service_date) AS last_date,
        COUNT(DISTINCT service_date) AS n_dates
    FROM gtfsrt
""").df().to_string(index=False))

## Diagnostic: which service IDs are active per date?

GTFS uses a `calendar` table keyed by day-of-week plus a `calendar_dates` table for one-off additions and removals (holidays, schedule changes). The query below resolves the active service IDs per date in the GTFS-RT archive. It should show 1-2 active services per weekday and a handful per weekend, matching SEPTA's weekday/Saturday/Sunday service pattern.

In [ ]:
print('Active service_ids per service_date:')
print(con.sql("""
    WITH dates AS (
        SELECT DISTINCT service_date FROM gtfsrt
    ),
    regular AS (
        SELECT d.service_date, c.service_id
        FROM dates d
        JOIN calendar c
          ON CAST(REPLACE(d.service_date, '-', '') AS INTEGER)
             BETWEEN c.start_date AND c.end_date
         AND CASE ISODOW(CAST(d.service_date AS DATE))
                 WHEN 1 THEN c.monday
                 WHEN 2 THEN c.tuesday
                 WHEN 3 THEN c.wednesday
                 WHEN 4 THEN c.thursday
                 WHEN 5 THEN c.friday
                 WHEN 6 THEN c.saturday
                 WHEN 7 THEN c.sunday
             END = 1
    ),
    additions AS (
        SELECT d.service_date, cd.service_id
        FROM dates d
        JOIN calendar_dates cd
          ON cd.date = CAST(REPLACE(d.service_date, '-', '') AS INTEGER)
         AND cd.exception_type = 1
    ),
    removals AS (
        SELECT d.service_date, cd.service_id
        FROM dates d
        JOIN calendar_dates cd
          ON cd.date = CAST(REPLACE(d.service_date, '-', '') AS INTEGER)
         AND cd.exception_type = 2
    ),
    combined AS (
        SELECT service_date, service_id FROM regular
        UNION
        SELECT service_date, service_id FROM additions
        EXCEPT
        SELECT service_date, service_id FROM removals
    )
    SELECT
        service_date,
        ISODOW(CAST(service_date AS DATE)) AS dow,
        COUNT(DISTINCT service_id) AS active_services
    FROM combined
    GROUP BY service_date
    ORDER BY service_date
""").df().to_string(index=False))

## Diagnostic: classifying day-service vs Owl trips

This project focuses on high-frequency daytime service. Routes 23 and 47 also run Owl service (after midnight, scheduled in static GTFS as 24:xx-27:xx times), but Owl is structurally different: different headways, different rider population, mostly different operational concerns.

The filter applied below is `first_stop_seconds < 86400`: trips whose first scheduled stop is before midnight. This keeps day-service trips even if their later stops bleed past midnight into 24:xx territory, which happens for late-evening trips.

In [ ]:
print('First-stop-time distribution for routes 23 and 47 trips:')
print(con.sql("""
    WITH trip_first_stop AS (
        SELECT
            t.trip_id,
            t.route_id,
            t.service_id,
            MIN(st.arrival_seconds) AS first_stop_seconds
        FROM trips t
        JOIN stop_times st USING (trip_id)
        WHERE t.route_id IN ('23', '47')
        GROUP BY t.trip_id, t.route_id, t.service_id
    )
    SELECT
        route_id,
        CASE
            WHEN first_stop_seconds < 14400  THEN '00_pre_4am'
            WHEN first_stop_seconds < 21600  THEN '01_early_AM_4_6'
            WHEN first_stop_seconds < 75600  THEN '02_high_freq_6_21'
            WHEN first_stop_seconds < 84600  THEN '03_late_eve_21_2330'
            WHEN first_stop_seconds < 86400  THEN '04_boundary_2330_24'
            WHEN first_stop_seconds < 100800 THEN '05_owl_24_28'
            ELSE                                  '06_owl_28_plus'
        END AS bucket,
        COUNT(*) AS n_trips,
        MIN(first_stop_seconds) AS min_secs,
        MAX(first_stop_seconds) AS max_secs
    FROM trip_first_stop
    GROUP BY route_id, bucket
    ORDER BY route_id, bucket
""").df().to_string(index=False))

## Build `matched_stops`

This is the intermediate table shared across all three snapshot rules. Every GTFS-RT prediction joined to its scheduled stop time, with all filters applied except dedup. One row per `(trip_id, stop_sequence, service_date, snapshot_ts)`, i.e., one prediction. A given stop event has many rows, one per snapshot it appeared in.

Filters applied:
- Routes 23 and 47
- `schedule_relationship = 0` (SCHEDULED)
- Day-service trips (first stop before midnight)
- Active service per date (with calendar_dates removals)

Two derived columns are added at the end:

- `scheduled_unix`: the predicted arrival time on the unix timeline. Includes a nearest-day correction for trips that bleed past midnight. If predictions cluster around a time closer to `midnight - 86400` or `midnight + 86400` than to `midnight`, we shift accordingly.
- `snapshot_unix`: `snapshot_ts` parsed as a unix epoch, used by the `last_before` and `latest` ordering rules.

In [ ]:
con.execute('DROP TABLE IF EXISTS matched_stops')

con.execute("""
    CREATE TABLE matched_stops AS
    WITH dates AS (
        SELECT DISTINCT service_date FROM gtfsrt
    ),
    active_services AS (
        SELECT d.service_date, c.service_id
        FROM dates d
        JOIN calendar c
          ON CAST(REPLACE(d.service_date, '-', '') AS INTEGER)
             BETWEEN c.start_date AND c.end_date
         AND CASE ISODOW(CAST(d.service_date AS DATE))
                 WHEN 1 THEN c.monday
                 WHEN 2 THEN c.tuesday
                 WHEN 3 THEN c.wednesday
                 WHEN 4 THEN c.thursday
                 WHEN 5 THEN c.friday
                 WHEN 6 THEN c.saturday
                 WHEN 7 THEN c.sunday
             END = 1

        UNION

        SELECT d.service_date, cd.service_id
        FROM dates d
        JOIN calendar_dates cd
          ON cd.date = CAST(REPLACE(d.service_date, '-', '') AS INTEGER)
         AND cd.exception_type = 1
    ),
    trip_first_stop AS (
        SELECT trip_id, MIN(arrival_seconds) AS first_stop_seconds
        FROM stop_times
        GROUP BY trip_id
    ),
    raw_matched AS (
        SELECT
            rt.trip_id,
            rt.route_id,
            rt.service_date,
            rt.snapshot_ts,
            rt.stop_sequence,
            rt.stop_id,
            rt.schedule_relationship,
            rt.predicted_unix,
            rt.predicted_ts,
            rt.midnight_unix,
            t.direction_id,
            st.arrival_time           AS scheduled_arrival_time,
            st.arrival_seconds        AS scheduled_arrival_seconds,
            st.departure_time         AS scheduled_departure_time,
            st.departure_seconds      AS scheduled_departure_seconds,
            st.timepoint
        FROM gtfsrt rt
        JOIN trips t
            ON rt.trip_id = t.trip_id
        JOIN active_services a
            ON t.service_id   = a.service_id
           AND a.service_date = rt.service_date
        JOIN stop_times st
            ON rt.trip_id        = st.trip_id
           AND rt.stop_sequence  = st.stop_sequence
        JOIN trip_first_stop tfs
            ON tfs.trip_id = rt.trip_id
        WHERE rt.route_id IN ('23', '47')
          AND rt.schedule_relationship = 0
          AND tfs.first_stop_seconds < 86400
          AND NOT EXISTS (
              SELECT 1 FROM calendar_dates cd
              WHERE cd.service_id     = t.service_id
                AND cd.date           = CAST(REPLACE(rt.service_date, '-', '') AS INTEGER)
                AND cd.exception_type = 2
          )
    )
    SELECT
        *,
        -- Nearest-day correction: pick the candidate scheduled_unix closest to predicted
        CASE
            WHEN ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds - 86400))
               < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds))
             AND ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds - 86400))
               < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds + 86400))
            THEN midnight_unix + scheduled_arrival_seconds - 86400
            WHEN ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds + 86400))
               < ABS(predicted_unix - (midnight_unix + scheduled_arrival_seconds))
            THEN midnight_unix + scheduled_arrival_seconds + 86400
            ELSE midnight_unix + scheduled_arrival_seconds
        END AS scheduled_unix,
        EPOCH(CAST(snapshot_ts AS TIMESTAMP)) AS snapshot_unix
    FROM raw_matched
""")

print('matched_stops built. Row count:')
print(con.sql('SELECT COUNT(*) AS rows FROM matched_stops').df().to_string(index=False))
print()
print('Distinct stop events (trip_id, stop_sequence, service_date):')
print(con.sql("""
    SELECT COUNT(*) AS distinct_stop_events
    FROM (
        SELECT trip_id, stop_sequence, service_date
        FROM matched_stops
        GROUP BY trip_id, stop_sequence, service_date
    )
""").df().to_string(index=False))

## Build the three `trip_delays_*` tables

Same `matched_stops` input, three different `ROW_NUMBER() OVER (PARTITION BY trip_id, stop_sequence, service_date ORDER BY ...)` orderings. Whichever row gets `snapshot_rank = 1` becomes the kept observation for that stop event.

Each table has the same schema: one row per scheduled stop event with a `delay_minutes` value. They differ only in *which* GTFS-RT observation produced that delay value.

In [ ]:
# closest: keep the prediction whose predicted_unix is closest to scheduled_unix
con.execute('DROP TABLE IF EXISTS trip_delays_closest')

con.execute("""
    CREATE TABLE trip_delays_closest AS
    WITH ranked AS (
        SELECT
            *,
            ROUND((predicted_unix - scheduled_unix) / 60.0, 1) AS delay_minutes,
            ROW_NUMBER() OVER (
                PARTITION BY trip_id, stop_sequence, service_date
                ORDER BY ABS(predicted_unix - scheduled_unix)
            ) AS snapshot_rank
        FROM matched_stops
    )
    SELECT
        trip_id, route_id, direction_id, service_date, snapshot_ts,
        stop_sequence, stop_id, schedule_relationship, timepoint,
        scheduled_arrival_time, scheduled_arrival_seconds,
        scheduled_departure_time, scheduled_departure_seconds,
        scheduled_unix, predicted_unix, predicted_ts,
        delay_minutes
    FROM ranked
    WHERE snapshot_rank = 1
""")

print('trip_delays_closest:')
print(con.sql('SELECT COUNT(*) AS rows FROM trip_delays_closest').df().to_string(index=False))

In [ ]:
# last_before: keep the latest snapshot whose snapshot_ts is BEFORE scheduled_unix.
# Some stop events have no qualifying snapshot (all their predictions arrived
# after scheduled time) and are dropped. Coverage diagnostic at the bottom.

con.execute('DROP TABLE IF EXISTS trip_delays_last_before')

con.execute("""
    CREATE TABLE trip_delays_last_before AS
    WITH eligible AS (
        SELECT *
        FROM matched_stops
        WHERE snapshot_unix < scheduled_unix
    ),
    ranked AS (
        SELECT
            *,
            ROUND((predicted_unix - scheduled_unix) / 60.0, 1) AS delay_minutes,
            ROW_NUMBER() OVER (
                PARTITION BY trip_id, stop_sequence, service_date
                ORDER BY snapshot_unix DESC
            ) AS snapshot_rank
        FROM eligible
    )
    SELECT
        trip_id, route_id, direction_id, service_date, snapshot_ts,
        stop_sequence, stop_id, schedule_relationship, timepoint,
        scheduled_arrival_time, scheduled_arrival_seconds,
        scheduled_departure_time, scheduled_departure_seconds,
        scheduled_unix, predicted_unix, predicted_ts,
        delay_minutes
    FROM ranked
    WHERE snapshot_rank = 1
""")

print('trip_delays_last_before:')
print(con.sql('SELECT COUNT(*) AS rows FROM trip_delays_last_before').df().to_string(index=False))

In [ ]:
# latest: keep the chronologically last snapshot for each stop event
con.execute('DROP TABLE IF EXISTS trip_delays_latest')

con.execute("""
    CREATE TABLE trip_delays_latest AS
    WITH ranked AS (
        SELECT
            *,
            ROUND((predicted_unix - scheduled_unix) / 60.0, 1) AS delay_minutes,
            ROW_NUMBER() OVER (
                PARTITION BY trip_id, stop_sequence, service_date
                ORDER BY snapshot_unix DESC
            ) AS snapshot_rank
        FROM matched_stops
    )
    SELECT
        trip_id, route_id, direction_id, service_date, snapshot_ts,
        stop_sequence, stop_id, schedule_relationship, timepoint,
        scheduled_arrival_time, scheduled_arrival_seconds,
        scheduled_departure_time, scheduled_departure_seconds,
        scheduled_unix, predicted_unix, predicted_ts,
        delay_minutes
    FROM ranked
    WHERE snapshot_rank = 1
""")

print('trip_delays_latest:')
print(con.sql('SELECT COUNT(*) AS rows FROM trip_delays_latest').df().to_string(index=False))

## Diagnostics: comparing the three rules

`closest` and `latest` should have identical row counts (both rank all snapshots). `last_before` will have fewer rows because it drops stop events where no snapshot arrived before scheduled time.

OTP under `closest` is consistently the highest (selection bias). `latest` is the lowest. `last_before` falls between.

In [ ]:
print('Row counts by snapshot rule:')
print(con.sql("""
    SELECT 'closest'      AS rule, COUNT(*) AS rows FROM trip_delays_closest
    UNION ALL
    SELECT 'last_before',          COUNT(*)         FROM trip_delays_last_before
    UNION ALL
    SELECT 'latest',               COUNT(*)         FROM trip_delays_latest
""").df().to_string(index=False))

print()
print('OTP by snapshot rule and route (-1 to +5 min window):')
print(con.sql("""
    WITH all_rules AS (
        SELECT 'closest'     AS rule, route_id, delay_minutes FROM trip_delays_closest
        UNION ALL
        SELECT 'last_before',         route_id, delay_minutes FROM trip_delays_last_before
        UNION ALL
        SELECT 'latest',              route_id, delay_minutes FROM trip_delays_latest
    )
    SELECT
        rule,
        route_id,
        COUNT(*) AS stop_events,
        ROUND(
            SUM(CASE WHEN delay_minutes BETWEEN -1 AND 5 THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 1
        ) AS otp_pct,
        ROUND(AVG(delay_minutes), 2) AS mean_delay,
        ROUND(MEDIAN(delay_minutes), 2) AS median_delay
    FROM all_rules
    GROUP BY rule, route_id
    ORDER BY route_id, rule
""").df().to_string(index=False))

In [ ]:
# Delay distribution by rule. Surfaces how the dedup choice shifts the shape
print('Delay distribution by snapshot rule:')
print(con.sql("""
    WITH all_rules AS (
        SELECT 'closest'     AS rule, delay_minutes FROM trip_delays_closest
        UNION ALL
        SELECT 'last_before',         delay_minutes FROM trip_delays_last_before
        UNION ALL
        SELECT 'latest',              delay_minutes FROM trip_delays_latest
    )
    SELECT
        rule,
        CASE
            WHEN delay_minutes < -60 THEN 'a_bad_match_neg'
            WHEN delay_minutes < -30 THEN 'b_very_early'
            WHEN delay_minutes < -1  THEN 'c_early'
            WHEN delay_minutes <= 5  THEN 'd_on_time'
            WHEN delay_minutes <= 15 THEN 'e_late'
            WHEN delay_minutes <= 60 THEN 'f_very_late'
            ELSE                          'g_bad_match_pos'
        END AS bucket,
        COUNT(*) AS records
    FROM all_rules
    GROUP BY rule, bucket
    ORDER BY rule, bucket
""").df().to_string(index=False))

In [ ]:
# How many stop events does last_before drop relative to closest?
print('Coverage of last_before relative to closest:')
print(con.sql("""
    WITH closest_events AS (
        SELECT trip_id, stop_sequence, service_date
        FROM trip_delays_closest
    ),
    last_before_events AS (
        SELECT trip_id, stop_sequence, service_date
        FROM trip_delays_last_before
    ),
    counts AS (
        SELECT
            (SELECT COUNT(*) FROM closest_events)     AS closest_n,
            (SELECT COUNT(*) FROM last_before_events) AS last_before_n
    )
    SELECT
        closest_n,
        last_before_n,
        closest_n - last_before_n AS dropped,
        ROUND(last_before_n * 100.0 / closest_n, 2) AS coverage_pct
    FROM counts
""").df().to_string(index=False))

In [ ]:
con.close()
print('Connection closed')